In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime
import logging        
from config import ROUTES, PipelineConfig  

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# Config 
BRONZE_PATH = "workspace.case_spark_cvm.bronze_fii_ativo_passivo_cvm"
NOME_TABELA  = f"silver_cvm_fii_ativo_passivo" 
SILVER_PATH = f"{ROUTES.TABLE_BASE}.{NOME_TABELA}"
DATA_PROC    = int(datetime.now().strftime("%Y%m%d"))

## CVM - Fundos Imobiliarios - Ativo Passivo

In [0]:
df_silver_fii_ativo_passivo = PipelineConfig.ler_ultima_particao(spark=spark, table_name=BRONZE_PATH, partition_col="data_processamento" )

### 1.1 tratemento silver

#### 1.1.1 Normalizando CNPJ

In [0]:
df_silver_fii_ativo_passivo = df_silver_fii_ativo_passivo.withColumn(
    "CNPJ_FUNDO_CLASSE",
    PipelineConfig.normalizar_cnpj("CNPJ_FUNDO_CLASSE")
)


#### 1.1.2 Retirando dados duplicados

In [0]:
# 1. Chaves que devem ser únicas
chave_negocio = ["CNPJ_FUNDO_CLASSE", "Data_Referencia"]

# 2. Removemos as duplicatas 
# Como não á uma regra clara de desempate ficamos com a linha com maior patrimonio liquido
df_silver_fii_ativo_passivo, df_quarentena_duplicadas = PipelineConfig.remover_duplicatas(
    df=df_silver_fii_ativo_passivo,
    chave_negocio=chave_negocio,
    coluna_ordenacao="Total_Investido" 
)

# 3. Salva a sujeira na quarentena
PipelineConfig.salvar_quarentena(
    spark=spark,
    df_quarentena=df_quarentena_duplicadas, 
    tabela_origem="bronze_fii_ativo_passivo_cvm", 
    data_proc=DATA_PROC
)


#### 1.1.3 Retirando dados nulos de Colunas Cores

In [0]:
df_silver_fii_ativo_passivo.display()

In [0]:
regras_qualidade = {
    "CNPJ_FUNDO_CLASSE": "not_null",  # Não pode ser vazio (Substitui o dropna)
    "Data_Referencia": "not_null",    # Não pode ser vazio (Substitui o dropna)
    "Total_Necessidades_Liquidez": "decimal",         # Não pode conter letras
    "Disponibilidades": "decimal",  # Não pode conter letras
    "Total_Investido": "decimal"   # Não pode conter letras
}

df_silver_fii_ativo_passivo, df_quarentena = PipelineConfig.aplicar_qualidade_e_separar(
    df=df_silver_fii_ativo_passivo,
    regras=regras_qualidade
    )

PipelineConfig.salvar_quarentena(
    spark=spark,
    df_quarentena=df_quarentena, 
    tabela_origem="bronze_fii_ativo_passivo_cvm", 
    data_proc=DATA_PROC
)

#### 1.1.4 Tratamento do Tipo de Dado

In [0]:
# Dropando as colunas de metadados
df_silver_fii_ativo_passivo = df_silver_fii_ativo_passivo.drop("_source_url", "_ingest_timestamp", "data_processamento")


In [0]:
df_silver_fii_ativo_passivo = df_silver_fii_ativo_passivo \
    .withColumn('cnpj_fundo_classe', f.col('CNPJ_FUNDO_CLASSE').cast(t.StringType())) \
    .withColumn('data_referencia', f.col('Data_Referencia').cast(t.DateType())) \
    .withColumn('versao', f.col('Versao').cast(t.IntegerType())) \
    .withColumn('total_necessidades_liquidez', f.col('Total_Necessidades_Liquidez').cast(t.DecimalType(22, 2))) \
    .withColumn('disponibilidades', f.col('Disponibilidades').cast(t.DecimalType(22, 2))) \
    .withColumn('titulos_publicos', f.col('Titulos_Publicos').cast(t.DecimalType(22, 2))) \
    .withColumn('titulos_privados', f.col('Titulos_Privados').cast(t.DecimalType(22, 2))) \
    .withColumn('fundos_renda_fixa', f.col('Fundos_Renda_Fixa').cast(t.DecimalType(18, 2))) \
    .withColumn('total_investido', f.col('Total_Investido').cast(t.DecimalType(18, 2))) \
    .withColumn('direitos_bens_imoveis', f.col('Direitos_Bens_Imoveis').cast(t.DecimalType(22, 2))) \
    .withColumn('terrenos', f.col('Terrenos').cast(t.DecimalType(18, 2))) \
    .withColumn('imoveis_renda_acabados', f.col('Imoveis_Renda_Acabados').cast(t.DecimalType(22, 2))) \
    .withColumn('imoveis_renda_construcao', f.col('Imoveis_Renda_Construcao').cast(t.DecimalType(22, 2))) \
    .withColumn('imoveis_venda_acabados', f.col('Imoveis_Venda_Acabados').cast(t.DecimalType(22, 2))) \
    .withColumn('imoveis_venda_construcao', f.col('Imoveis_Venda_Construcao').cast(t.DecimalType(22, 2))) \
    .withColumn('outros_direitos_reais', f.col('Outros_Direitos_Reais').cast(t.DecimalType(22, 2))) \
    .withColumn('acoes', f.col('Acoes').cast(t.DecimalType(22, 2))) \
    .withColumn('debentures', f.col('Debentures').cast(t.DecimalType(22, 2))) \
    .withColumn('bonus_subscricao', f.col('Bonus_Subscricao').cast(t.DecimalType(22, 2))) \
    .withColumn('certificados_deposito_valores_mobiliarios', f.col('Certificados_Deposito_Valores_Mobiliarios').cast(t.DecimalType(22, 2))) \
    .withColumn('cedulas_debentures', f.col('Cedulas_Debentures').cast(t.DecimalType(12, 2))) \
    .withColumn('fundo_acoes', f.col('Fundo_Acoes').cast(t.DecimalType(22, 2))) \
    .withColumn('fip', f.col('FIP').cast(t.DecimalType(22, 2))) \
    .withColumn('fii', f.col('FII').cast(t.DecimalType(22, 2))) \
    .withColumn('fdic', f.col('FDIC').cast(t.DecimalType(22, 2))) \
    .withColumn('outras_cotas_fi', f.col('Outras_Cotas_FI').cast(t.DecimalType(22, 2))) \
    .withColumn('notas_promissorias', f.col('Notas_Promissorias').cast(t.DecimalType(22, 2))) \
    .withColumn('acoes_sociedades_atividades_fii', f.col('Acoes_Sociedades_Atividades_FII').cast(t.DecimalType(22, 2))) \
    .withColumn('cotas_sociedades_atividades_fii', f.col('Cotas_Sociedades_Atividades_FII').cast(t.DecimalType(22, 2))) \
    .withColumn('cepac', f.col('CEPAC').cast(t.DecimalType(22, 2))) \
    .withColumn('cri', f.col('CRI').cast(t.DecimalType(22, 2))) \
    .withColumn('cri_cra', f.col('CRI_CRA').cast(t.DecimalType(22, 2))) \
    .withColumn('letras_hipotecarias', f.col('Letras_Hipotecarias').cast(t.DecimalType(22, 2))) \
    .withColumn('lci', f.col('LCI').cast(t.DecimalType(22, 2))) \
    .withColumn('lci_lca', f.col('LCI_LCA').cast(t.DecimalType(22, 2))) \
    .withColumn('lig', f.col('LIG').cast(t.DecimalType(22, 2))) \
    .withColumn('outros_valores_mobliarios', f.col('Outros_Valores_Mobliarios').cast(t.DecimalType(22, 2))) \
    .withColumn('valores_receber', f.col('Valores_Receber').cast(t.DecimalType(22, 2))) \
    .withColumn('contas_receber_aluguel', f.col('Contas_Receber_Aluguel').cast(t.DecimalType(22, 2))) \
    .withColumn('contas_receber_venda_imoveis', f.col('Contas_Receber_Venda_Imoveis').cast(t.DecimalType(22, 2))) \
    .withColumn('outros_valores_receber', f.col('Outros_Valores_Receber').cast(t.DecimalType(22, 2))) \
    .withColumn('rendimentos_distribuir', f.col('Rendimentos_Distribuir').cast(t.DecimalType(22, 2))) \
    .withColumn('taxa_administracao_pagar', f.col('Taxa_Administracao_Pagar').cast(t.DecimalType(22, 2))) \
    .withColumn('taxa_performance_pagar', f.col('Taxa_Performance_Pagar').cast(t.DecimalType(22, 2))) \
    .withColumn('obrigacoes_aquisicao_imoveis', f.col('Obrigacoes_Aquisicao_Imoveis').cast(t.DecimalType(22, 2))) \
    .withColumn('adiantamento_venda_imoveis', f.col('Adiantamento_Venda_Imoveis').cast(t.DecimalType(22, 2))) \
    .withColumn('adiantamento_alugueis', f.col('Adiantamento_Alugueis').cast(t.DecimalType(22, 2))) \
    .withColumn('obrigacoes_securitizacao_recebiveis', f.col('Obrigacoes_Securitizacao_Recebiveis').cast(t.DecimalType(22, 2))) \
    .withColumn('instrumentos_financeiros_derivativos', f.col('Instrumentos_Financeiros_Derivativos').cast(t.DecimalType(22, 2))) \
    .withColumn('provisoes_contigencias', f.col('Provisoes_Contigencias').cast(t.DecimalType(22, 2))) \
    .withColumn('outros_valores_pagar', f.col('Outros_Valores_Pagar').cast(t.DecimalType(22, 2))) \
    .withColumn('total_passivo', f.col('Total_Passivo').cast(t.DecimalType(22, 2))) 

### 1.2 Salvar na camada Silver

In [0]:
# Definindo as chaves estrangeiras 
chave_negocio = ["cnpj_fundo_classe", "data_referencia"]

PipelineConfig.upsert_silver(
    spark=spark, 
    df_novo=df_silver_fii_ativo_passivo, 
    tabela_destino=SILVER_PATH, 
    chave_negocio=chave_negocio
    )

In [0]:
%sql
select 
    *
from  workspace.case_spark_cvm.silver_cvm_fii_ativo_passivo


In [0]:
%sql
select 
    *
from workspace.case_spark_cvm.silver_quarentena